# धडा 04 - टूल यूज डिझाइन पॅटर्न

या धड्यात तुम्ही Microsoft Agent Framework (Python) वापरून AI एजंटसाठी **टूल यूज** डिझाइन पॅटर्न शिकणार आहात. आम्ही यामध्ये समाविष्ट आहोत:

- `@tool` डेकोरेटर आणि टाईप केलेल्या पॅरामीटर्ससह फंक्शन टूल्सची व्याख्या करणे
- मॉडेलला प्रत्येक टूल काय करते हे कळण्यासाठी टूल स्कीमा प्रदान करणे
- `approval_mode` वापरून टूल अंमलबजावणीवर नियंत्रण ठेवणे
- Pydantic मॉडेल आणि `response_format` द्वारे **संरचित आउटपुट** परत करणे

परिस्थिती अशी आहे की एक **प्रवास बुकिंग एजंट** जो डेस्टिनेशन शोधू शकतो, उपलब्धता तपासू शकतो, आणि फ्लाइट माहिती मिळवू शकतो.


## सेटअप


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## @tool डेकोरेटरसह टूल्सची व्याख्या करणे

`@tool` डेकोरेटर साध्या Python फंक्शनला एका टूलमध्ये परिवर्तित करतो ज्याला एजंट कॉल करू शकतो.
महत्‍वाचे मुद्दे:

- **docstring** हे टूलचे वर्णन होते जे मॉडेल पाहते.
- **टाइप अ‍ॅनोटेशन्स** (वर्णनांसह `Annotated` सह) टूल स्कीमा परिभाषित करतात.
- `approval_mode` नियंत्रित करते की वापरकर्त्याने प्रत्येक कॉलचे अनुमोदन करणे आवश्यक आहे की नाही.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get available vacation destinations."""
    return ["Barcelona", "Paris", "Berlin", "Tokyo", "Sydney", "New York City"]


@tool(approval_mode="never_require")
def check_availability(
    destination: Annotated[str, "The destination to check"],
) -> str:
    """Check booking availability for a destination."""
    availability = {
        "Barcelona": "Available - 3 spots left",
        "Paris": "Available",
        "Berlin": "Sold out",
        "Tokyo": "Available - 1 spot left",
        "Sydney": "Available",
        "New York City": "Available",
    }
    return availability.get(destination, "Unknown destination")


@tool(approval_mode="never_require")
def get_flight_info(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
) -> str:
    """Get flight information between two cities."""
    flights = {
        "LHR-BCN": "BA 2042, Departs 08:30, Arrives 11:45, $350",
        "LHR-CDG": "AF 1081, Departs 09:15, Arrives 11:30, $280",
        "LHR-NRT": "JL 044, Departs 11:00, Arrives 07:00+1, $890",
    }
    return flights.get(
        f"{origin}-{destination}",
        f"No direct flights from {origin} to {destination}",
    )

## एकाधिक साधनांसह एजंट तयार करणे

मॉडेल वापरकर्त्याच्या प्रश्नाचे उत्तर देण्यासाठी आवश्यक असलेली कोणतीही साधने कॉल करू शकेल यासाठी सर्व तीन साधने क्लायंटला द्या.


In [ ]:
travel_tools = [get_destinations, check_availability, get_flight_info]

agent = client.as_agent(
    name="TravelToolAgent",
    instructions="You are a travel agent. Use the available tools to answer questions about destinations, availability, and flights.",
    tools=travel_tools,
)

response = await agent.run(
    "What destinations do you have? Which ones are still available?"
)
print(response)

## टूल्ससह संरचित उत्पादन

`response_format` ला Pydantic मॉडेल म्हणून सेट केल्याने, एजंटला मुक्त-रूपातील मजकूराऐवजी चांगल्या प्रकारे टाइप केलेला JSON ऑब्जेक्ट परत करावा लागतो. जेव्हा खालील कोडला निकाल प्रोग्रामॅटिक पद्धतीने वापरावा लागतो तेव्हा हे उपयुक्त ठरते.


In [ ]:
class BookingRecommendation(BaseModel):
    destination: str
    available: bool
    flight_details: str
    estimated_cost: int


class TravelPlan(BaseModel):
    recommendations: list[BookingRecommendation]


structured_agent = client.as_agent(
    name="StructuredTravelAgent",
    instructions=(
        "You are a travel agent. Use the available tools to find destinations, "
        "check availability, and get flight info. Return structured results."
    ),
    tools=[get_destinations, check_availability, get_flight_info],
)

response = await structured_agent.run(
    "I want to fly from London Heathrow to somewhere warm in Europe. "
    "Check what's available."
)
if response:
    print(response)

## टूल मंजुरी नमुने

`@tool` वरील `approval_mode` पॅरामीटर नियंत्रित करते की टूल कॉल्सना अंमलबजावणीपूर्वी मानवी मंजुरी आवश्यक आहे की नाही:

| मोड | वर्तन |
|---|---|
| `"never_require"` | टूल आपोआप चालते — वापरकर्त्याची पुष्टी आवश्यक नाही. |
| `"always_require"` | प्रत्येक कॉल चालवण्यापूर्वी वापरकर्त्याने मंजुरी दिली पाहिजे. |

साइड-इफेक्ट असलेल्या टूल्ससाठी `"always_require"` वापरा (उदा. फ्लाइट बुकिंग, क्रेडिट कार्ड चार्ज करणे) जेणेकरून मानवी सहभाग स्थितीत राहील.


In [ ]:
@tool(approval_mode="always_require")
def book_flight(
    origin: Annotated[str, "Origin airport code"],
    destination: Annotated[str, "Destination airport code"],
    passenger_name: Annotated[str, "Full name of the passenger"],
) -> str:
    """Book a flight for a passenger. Requires approval before executing."""
    return (
        f"Flight booked from {origin} to {destination} "
        f"for {passenger_name}. Confirmation #TRV-2024-{hash(passenger_name) % 10000:04d}"
    )


print("Tool name:", book_flight.name)
print("Approval mode:", book_flight.approval_mode)

## सारांश

या धड्यात आपण शिकले कसे:

1. **टूल्स व्याख्यित करणे** `@tool` डेकोरेटर वापरून टायप केलेल्या पॅरामीटर्स आणि डॉकस्ट्रिंगसह जे टूल स्कीमा म्हणून वापरले जातात.
2. **अनेक टूल्स संकलित करणे** जेणेकरून एजंट त्यांना क्रमाने कॉल करू शकतो आणि गुंतागुंतीच्या प्रश्नांची उत्तरे देऊ शकतो.
3. **रचनेत आउटपुट परत करणे** Pydantic मॉडेल `response_format` म्हणून पास करून.
4. **टूल मंजुरी नियंत्रण** `approval_mode` वापरून जेणेकरून संवेदनशील क्रियांसाठी माणूस प्रक्रियेत राहील.

हे पॅटर्न विश्वसनीय, उत्पादन-तयार एजंट तयार करण्यासाठी पाया तयार करतात जे सुरक्षितपणे बाह्य प्रणालींशी संवाद साधू शकतात.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
हा दस्तऐवज AI भाषांतर सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) चा वापर करून अनुवादित केला आहे. जरी आम्ही अचूकतेसाठी प्रयत्न करतो, तरी कृपया लक्षात घ्या की स्वयंचलित भाषांतरांमध्ये त्रुटी किंवा अचूकतेची कमतरता असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला पाहिजे. महत्त्वाची माहिती असल्यास, व्यावसायिक मानवी भाषांतराची शिफारस केली जाते. या भाषांतराच्या वापरामुळे उद्भवणाऱ्या कोणत्याही गैरसमज किंवा चुकीच्या अर्थलावणीसाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
